# TP 2 — Construire l'auto-attention pas à pas

Le but ici c'est de partir d'une moyenne basique des tokens précédents pour arriver à une vraie couche d'attention où le modèle apprend lui-même sur quels tokens il doit se concentrer.

La règle d'or : chaque ligne de poids représente l'influence des tokens passés sur le token actuel, et le total de la ligne doit toujours faire 1.

In [8]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 : agréger par produit matriciel

In [9]:
torch.manual_seed(42)

matrice_a = torch.tril(torch.ones(3, 3))
matrice_a = matrice_a / torch.sum(matrice_a, 1, keepdim=True)

matrice_b = torch.randint(0, 10, (3, 2)).float()
resultat = matrice_a @ matrice_b

print(matrice_a)
print(matrice_b)
print(resultat)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


**Questions d'interprétation** — Sans Python, calculez c₂. Pourquoi la somme de chaque ligne de `a` vaut-elle 1 ?

**Réponses :**
- Si on le fait à la main pour c2 (qui est la deuxième ligne) : c'est la moyenne de la ligne 1 et 2 de b. Donc (2+6)/2 = 4, et (7+4)/2 = 5.5. On retrouve bien [4.0, 5.5]. Pour la troisième ligne on divise par 3, etc.
- La somme des lignes fait 1 parce qu'on a divisé chaque ligne par sa propre somme. Ça permet de faire une vraie moyenne des vecteurs qu'on a vus jusqu'ici, plutôt qu'une simple addition qui exploserait au fur et à mesure qu'on avance dans le temps.

## Étape 2 : moyenner uniquement le passé

In [10]:
torch.manual_seed(1337)
batch_size, temps, dim = 4, 8, 2
x = torch.randn(batch_size, temps, dim)

x_moyen = torch.zeros((batch_size, temps, dim))
for b in range(batch_size):
    for t in range(temps):
        historique = x[b, :t + 1]
        x_moyen[b, t] = torch.mean(historique, dim=0)

print(historique)

tensor([[ 1.6455, -0.8030],
        [ 1.3514, -0.2759],
        [-1.5108,  2.1048],
        [ 2.7630, -1.7465],
        [ 1.4516, -1.5103],
        [ 0.8212, -0.2115],
        [ 0.7789,  1.5333],
        [ 1.6097, -0.4032]])


**Questions d'interprétation** — Donnez les formes de `xprev` pour t=0, t=3 et t=7. Pourquoi la moyenne porte-t-elle sur `dim=0` ?

**Réponses :**
- Pour l'historique, la forme c'est (t+1, dim). Donc pour t=0 on a (1, 2). Pour t=3 c'est (4, 2) et pour t=7 c'est (8, 2).
- On fait la moyenne sur dim=0 parce que c'est la dimension du temps (les tokens). On veut écraser cette dimension pour garder un seul vecteur de taille 2 qui résume tout ce qu'on a vu avant.

## Étapes 3–4 : vectoriser

Au lieu de faire des boucles for (ce qui est super lent), on va utiliser une matrice triangulaire pour faire la même chose d'un coup.

### Version 2 : produit matriciel

In [11]:
poids_att = torch.tril(torch.ones(temps, temps))
poids_att = poids_att / poids_att.sum(1, keepdim=True)
x_moyen2 = poids_att @ x

print("Même résultat ?", torch.allclose(x_moyen, x_moyen2, atol=1e-6))

Même résultat ? True


### Version 3 : masque + Softmax

In [12]:
masque = torch.tril(torch.ones(temps, temps))
poids_softmax = torch.zeros((temps, temps))
poids_softmax = poids_softmax.masked_fill(masque == 0, float('-inf'))
poids_softmax = F.softmax(poids_softmax, dim=-1)

x_moyen3 = poids_softmax @ x
print("Même résultat ?", torch.allclose(x_moyen, x_moyen3, atol=1e-6))
print(poids_softmax)

Même résultat ? True
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


**Questions d'interprétation** — Pourquoi `wei @ x` fonctionne-t-il avec une matrice (T, T) et un tenseur (B, T, C) ? Que devient Softmax aux emplacements contenant −∞ ?

**Réponses :**
- Ça marche grâce au broadcasting de PyTorch. L'opérateur @ va faire la multiplication matricielle sur les deux dernières dimensions et juste répéter la matrice des poids (T, T) pour chaque élément du batch B.
- L'exponentielle de moins l'infini fait 0. Du coup le softmax met un poids de 0 absolu sur toutes ces cases. C'est ça qui crée notre masque causal et qui empêche formellement le modèle de tricher en regardant les tokens du futur.

## Étape 5 : calculer des poids d'attention

In [13]:
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
taille_tete = 16

couche_cle = nn.Linear(C, taille_tete, bias=False)
couche_requete = nn.Linear(C, taille_tete, bias=False)
couche_valeur = nn.Linear(C, taille_tete, bias=False)

cle = couche_cle(x)
requete = couche_requete(x)

scores = requete @ cle.transpose(-2, -1)
scores = scores * (taille_tete ** -0.5)
scores = scores.masked_fill(masque == 0, float('-inf'))
scores = F.softmax(scores, dim=-1)

valeur = couche_valeur(x)
sortie_attention = scores @ valeur

**Questions d'interprétation** — Quelle ligne de `wei[0]` correspond au dernier token ? Pourquoi la sortie mélange-t-elle v, et non directement k ?

**Réponses :**
- Le dernier token est sur la toute dernière ligne (l'indice 7). C'est logique puisque c'est la seule ligne qui n'a aucun 0, ce token peut voir tous les autres avant lui.
- On mélange 'v' (la valeur) et pas 'k' (la clé) parce qu'ils ne servent pas à la même chose. 'q' et 'k' servent juste à calculer le score de match entre deux tokens. Une fois qu'on sait à quel point ils matchent, c'est le vecteur 'v' qui contient la vraie information qu'on veut faire passer.

## Étape 6 : stabiliser Softmax

In [14]:
k_test = torch.randn(B, T, taille_tete)
q_test = torch.randn(B, T, taille_tete)

score_brut = q_test @ k_test.transpose(-2, -1)
score_ajuste = score_brut * (taille_tete ** -0.5)

print("Variance avant:", score_brut.var().item())
print("Variance après:", score_ajuste.var().item())

z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print(torch.softmax(z, dim=-1))
print(torch.softmax(z * 8, dim=-1))

Variance avant: 19.847789764404297
Variance après: 1.2404868602752686
tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


**Bilan du TP 2** — Vérifiez la somme des lignes de `wei`, l'absence de poids vers le futur et la forme finale. Expliquez le rôle distinct de Q, K et V.

**Bilan :**
- Si on fait la somme des lignes de notre matrice de scores, ça fait bien 1 partout grâce au softmax.
- Le triangle supérieur de la matrice est à 0 donc aucune info du futur ne fuite.
- L'output a bien la forme (4, 8, 16) soit (Batch, Temps, Taille_Tete).
- En résumé pour Q, K et V :
  - Q (Query) : c'est ce que le token actuel cherche comme info.
  - K (Key) : c'est l'info que chaque token prétend avoir (un peu comme un mot-clé).
  - V (Value) : c'est le contenu réel qu'on transmet si Q et K matchent bien.
- Pour l'étape de stabilisation : la variance de base tourne autour de 16 (la taille de notre tête), et quand on divise par la racine carrée, on la ramène vers 1. C'est super important parce que si on laisse la variance exploser, le softmax va devenir hyper binaire (comme on le voit dans le dernier print où une valeur prend 80% du poids d'un coup) et ça bloque l'apprentissage car les gradients s'annulent.